# AMD — análise financeira

Notebook para Google Colab: obtém dados históricos da AMD, apresenta demonstrações financeiras e cria gráficos com uma apresentação mais cuidada.

**Não exporta CSV nem ficheiros.** Os resultados são apresentados diretamente no notebook.

> Dados via Yahoo Finance (`yfinance`), sujeitos a disponibilidade e atraso. Conteúdo informativo, não aconselhamento financeiro.

In [ ]:
%pip -q install yfinance pandas matplotlib

import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from pathlib import Path

TICKER = "AMD"
PERIODO = "5y"       # exemplos: "1y", "2y", "5y", "max"
INTERVALO = "1d"     # exemplos: "1d", "1wk", "1mo"

# Formato vertical 4:5 para Instagram (1080 × 1350 px).
# As imagens PNG são guardadas na pasta instagram_amd/; não são criados CSV.
PASTA_IMAGENS = Path("instagram_amd")
PASTA_IMAGENS.mkdir(exist_ok=True)
COR_FUNDO = "#0B1020"
COR_TEXTO = "#F8FAFC"
COR_SECUNDARIA = "#A8B3C7"
COR_AZUL = "#5B8CFF"
COR_VERDE = "#35D0A0"
COR_LARANJA = "#FFB454"

plt.rcParams.update({
    "figure.facecolor": COR_FUNDO,
    "axes.facecolor": COR_FUNDO,
    "text.color": COR_TEXTO,
    "axes.labelcolor": COR_SECUNDARIA,
    "xtick.color": COR_SECUNDARIA,
    "ytick.color": COR_SECUNDARIA,
    "font.family": "DejaVu Sans",
})

amd = yf.Ticker(TICKER)
precos = amd.history(period=PERIODO, interval=INTERVALO, auto_adjust=False)
if precos.empty:
    raise RuntimeError("Sem preços disponíveis. Confirma a ligação e os parâmetros.")

print(f"Dados carregados: {TICKER} · {PERIODO} · {len(precos):,} registos")
display(precos.tail())

## 1. Evolução do preço da ação

In [ ]:
# POST 1 — Preço da ação: proporção 4:5, 1080 × 1350 px
fecho = precos["Close"].dropna()
fig, ax = plt.subplots(figsize=(8, 10), dpi=135)
fig.subplots_adjust(left=0.13, right=0.91, top=0.79, bottom=0.18)
fig.text(0.10, 0.93, "AMD  /  MARKET WATCH", fontsize=11, color=COR_VERDE, weight="bold")
fig.text(0.10, 0.875, "A ação em perspetiva", fontsize=25, weight="bold")
fig.text(0.10, 0.835, f"Preço de fecho · últimos {PERIODO}", fontsize=12, color=COR_SECUNDARIA)
ax.plot(fecho.index, fecho, color=COR_AZUL, linewidth=3)
ax.fill_between(fecho.index, fecho.astype(float), color=COR_AZUL, alpha=0.18)
ax.scatter(fecho.index[-1], fecho.iloc[-1], color=COR_VERDE, s=70, zorder=4, edgecolors=COR_FUNDO, linewidth=1.5)
ax.annotate(f"${fecho.iloc[-1]:.2f}", (fecho.index[-1], fecho.iloc[-1]),
            xytext=(-10, 16), textcoords="offset points", ha="right",
            fontsize=16, color=COR_VERDE, weight="bold")
ax.set_ylabel("Preço (USD)", fontsize=11)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.spines["bottom"].set_color("#29334A")
ax.grid(axis="y", color="#29334A", alpha=0.8, linewidth=0.8)
ax.tick_params(axis="both", labelsize=9, length=0, pad=8)
ax.set_facecolor(COR_FUNDO)
fig.text(0.10, 0.095, "Fonte: Yahoo Finance via yfinance", fontsize=9, color=COR_SECUNDARIA)
fig.text(0.10, 0.065, "Conteúdo informativo · Não é aconselhamento financeiro", fontsize=8, color=COR_SECUNDARIA)
fig.savefig(PASTA_IMAGENS / "amd_preco_instagram.png", dpi=135, facecolor=COR_FUNDO)
plt.show()
print("Imagem guardada em:", PASTA_IMAGENS / "amd_preco_instagram.png")

## 2. Volume negociado

In [ ]:
# POST 2 — Volume negociado, formato Instagram 4:5
fig, ax = plt.subplots(figsize=(8, 10), dpi=135)
fig.subplots_adjust(left=0.13, right=0.91, top=0.79, bottom=0.18)
fig.text(0.10, 0.93, "AMD  /  MARKET WATCH", fontsize=11, color=COR_VERDE, weight="bold")
fig.text(0.10, 0.875, "Atividade do mercado", fontsize=25, weight="bold")
fig.text(0.10, 0.835, f"Volume negociado · últimos {PERIODO}", fontsize=12, color=COR_SECUNDARIA)
ax.fill_between(precos.index, precos["Volume"], color=COR_VERDE, alpha=0.68, linewidth=0)
ax.plot(precos.index, precos["Volume"], color=COR_VERDE, linewidth=0.8, alpha=0.9)
ax.set_ylabel("Ações negociadas", fontsize=11)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.spines["bottom"].set_color("#29334A")
ax.grid(axis="y", color="#29334A", alpha=0.8, linewidth=0.8)
ax.tick_params(axis="both", labelsize=9, length=0, pad=8)
fig.text(0.10, 0.095, "Fonte: Yahoo Finance via yfinance", fontsize=9, color=COR_SECUNDARIA)
fig.text(0.10, 0.065, "Conteúdo informativo · Não é aconselhamento financeiro", fontsize=8, color=COR_SECUNDARIA)
fig.savefig(PASTA_IMAGENS / "amd_volume_instagram.png", dpi=135, facecolor=COR_FUNDO)
plt.show()
print("Imagem guardada em:", PASTA_IMAGENS / "amd_volume_instagram.png")

## 3. Demonstrações financeiras

In [ ]:
demonstracoes = {
    "Resultados anuais": amd.income_stmt,
    "Resultados trimestrais": amd.quarterly_income_stmt,
    "Balanço anual": amd.balance_sheet,
    "Balanço trimestral": amd.quarterly_balance_sheet,
    "Fluxo de caixa anual": amd.cashflow,
    "Fluxo de caixa trimestral": amd.quarterly_cashflow,
}
for nome, tabela in demonstracoes.items():
    print(f"\n### {nome}")
    if tabela is not None and not tabela.empty:
        display(tabela)
    else:
        print("Sem dados disponíveis.")

## 4. Receita e lucro líquido

In [ ]:
# POST 3 — Receita e lucro líquido anual, formato Instagram 4:5
resultados = amd.income_stmt
if resultados is not None and not resultados.empty:
    linhas = {str(indice).lower(): indice for indice in resultados.index}
    receita_key = next((v for k, v in linhas.items() if "total revenue" in k or k == "revenue"), None)
    lucro_key = next((v for k, v in linhas.items() if "net income" in k), None)
    series = {}
    if receita_key is not None:
        series["Receita"] = pd.to_numeric(resultados.loc[receita_key], errors="coerce")
    if lucro_key is not None:
        series["Lucro líquido"] = pd.to_numeric(resultados.loc[lucro_key], errors="coerce")
    if series:
        financeiro = pd.DataFrame(series).sort_index()
        display(financeiro)
        fig, ax = plt.subplots(figsize=(8, 10), dpi=135)
        fig.subplots_adjust(left=0.15, right=0.91, top=0.79, bottom=0.20)
        fig.text(0.10, 0.93, "AMD  /  FUNDAMENTALS", fontsize=11, color=COR_VERDE, weight="bold")
        fig.text(0.10, 0.875, "Receita & resultado", fontsize=25, weight="bold")
        fig.text(0.10, 0.835, "Demonstrações anuais · USD", fontsize=12, color=COR_SECUNDARIA)
        financeiro.plot(kind="bar", ax=ax, color=[COR_AZUL, COR_LARANJA][:len(financeiro.columns)], width=0.68)
        ax.set_xlabel("")
        ax.set_ylabel("USD", fontsize=11)
        ax.spines[["top", "right", "left"]].set_visible(False)
        ax.spines["bottom"].set_color("#29334A")
        ax.grid(axis="y", color="#29334A", alpha=0.8, linewidth=0.8)
        ax.tick_params(axis="both", labelsize=9, length=0, pad=8)
        ax.ticklabel_format(axis="y", style="plain")
        ax.legend(frameon=False, labelcolor=COR_TEXTO)
        plt.xticks(rotation=0)
        fig.text(0.10, 0.095, "Fonte: Yahoo Finance via yfinance", fontsize=9, color=COR_SECUNDARIA)
        fig.text(0.10, 0.065, "Conteúdo informativo · Não é aconselhamento financeiro", fontsize=8, color=COR_SECUNDARIA)
        fig.savefig(PASTA_IMAGENS / "amd_receita_lucro_instagram.png", dpi=135, facecolor=COR_FUNDO)
        plt.show()
        print("Imagem guardada em:", PASTA_IMAGENS / "amd_receita_lucro_instagram.png")
    else:
        print("Não foi possível identificar receita/lucro nos dados disponíveis.")
else:
    print("Demonstração de resultados não disponível.")

## 5. Indicadores gerais

In [ ]:
try:
    info = amd.get_info()
    chaves = ["longName", "sector", "industry", "currency", "marketCap",
              "trailingPE", "forwardPE", "priceToBook", "profitMargins",
              "returnOnEquity", "dividendYield"]
    resumo = {chave: info.get(chave) for chave in chaves if info.get(chave) is not None}
    display(pd.DataFrame(resumo.items(), columns=["Indicador", "Valor"]))
except Exception as erro:
    print(f"Não foi possível obter os indicadores: {erro}")